In [150]:
!wget https://osf.io/download/bphvt/ -O atb.metadata.202505.sqlite.xz

--2026-02-13 15:00:34--  https://osf.io/download/bphvt/
Resolving osf.io (osf.io)... 35.190.84.173
Connecting to osf.io (osf.io)|35.190.84.173|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://files.de-1.osf.io/v1/resources/h7wzy/providers/osfstorage/6862a7dc04d1e726e7128a9e [following]
--2026-02-13 15:00:34--  https://files.de-1.osf.io/v1/resources/h7wzy/providers/osfstorage/6862a7dc04d1e726e7128a9e
Resolving files.de-1.osf.io (files.de-1.osf.io)... 35.186.249.111
Connecting to files.de-1.osf.io (files.de-1.osf.io)|35.186.249.111|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://storage.googleapis.com/cos-osf-prod-files-de-1/5d6ff9a95516d2269f52fa884d29179b74b3c9acf0c5511e6be87dd06a5af9c0?response-content-disposition=attachment%3B%20filename%3D%22atb.metadata.202505.sqlite.xz%22%3B%20filename%2A%3DUTF-8%27%27atb.metadata.202505.sqlite.xz&GoogleAccessId=files-de-1%40cos-osf-prod.iam.gserviceaccount.com&Expires=17

In [151]:
!xz -d atb.metadata.202505.sqlite.xz

In [201]:
import sqlite3
import polars as pl


uri = "sqlite://atb.metadata.202505.sqlite"
query = "SELECT sample_accession, Species, Sequence_abundance FROM sylph;"

df = pl.read_database_uri(query=query, uri=uri)

In [202]:
cf_path = (
    df
        .filter(
            (pl.col('Species') == 'Staphylococcus aureus')
            | (pl.col('Species') == 'Pseudomonas aeruginosa') 
            | (pl.col('Species') == 'Haemophilus influenzae') 
            | (pl.col('Species') == 'Stenotrophomonas maltophilia') 
            | (pl.col('Species').str.contains('Burkholderia')) 
            | (pl.col('Species').str.contains('Achromobacter'))
        )
)

In [206]:
# some have duplicates, so only consider the most abundant one
(
    cf_path
        .sort('Sequence_abundance', descending=True)
        .unique('sample_accession',  maintain_order=True)
        .write_csv('cf_pathogen_biosamples.csv')
)